# 01 · Datos, partición y auditoría
**Proyecto Final Integrador — TC3009C.** Este notebook ejecuta el protocolo obligatorio de la Sección 3,
reporta el *Test Audit ID* y verifica las reglas de integridad experimental (cero fuga de datos,
aislamiento de test y normalización simétrica para BLEU).

In [ ]:
# === 0. Entorno: Colab (GPU) o local =========================================
# En Colab: Entorno de ejecución -> Cambiar tipo -> GPU (T4 o superior).
REPO_URL = "https://github.com/bgoni04/nmt-en-es-tc3009c.git"   # <- CAMBIAR por su repo
USE_DRIVE = False   # guarda checkpoints/resultados en Drive para usarlos en el notebook 06

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("nmt-en-es-tc3009c"):
        !git clone -q {REPO_URL}
    %cd nmt-en-es-tc3009c
    !pip install -q -r requirements.txt
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["NMT_RESULTS_DIR"] = "/content/drive/MyDrive/nmt-en-es-tc3009c/results"
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src.utils import set_seed, get_device, gpu_name, results_dir
set_seed(23)
DEVICE = get_device()
RESULTS = results_dir()
print("Dispositivo:", DEVICE, "|", gpu_name(), "| resultados en:", RESULTS)

In [ ]:
# === 1. Descarga directa (Sección 3 del enunciado) ============================
import os
if not os.path.exists("spa.txt"):
    !wget -q https://www.manythings.org/anki/spa-eng.zip && unzip -q -o spa-eng.zip
if not os.path.exists("spa.txt"):   # manythings a veces rechaza el user-agent de wget
    !wget -q -U "Mozilla/5.0" https://www.manythings.org/anki/spa-eng.zip && unzip -q -o spa-eng.zip
assert os.path.exists("spa.txt"), "No se pudo descargar spa.txt"

In [ ]:
# === 2. Protocolo OBLIGATORIO de carga, filtrado y partición (Listing 1, sin cambios) ===
import random, hashlib

with open('spa.txt', 'r', encoding='utf-8') as f:
    raw_pairs = [line.strip().split('\t')[:2] for line in f if '\t' in line]

# 2. Filtrado de complejidad mínima (>= 8 palabras en inglés, <= 120 tokens)
filtered_pairs = [
    (en, es) for en, es in raw_pairs
    if 8 <= len(en.split()) <= 120 and len(es.split()) <= 120
]

# 3. Mezcla determinista con semilla fija (obligatoria para auditoría)
rng = random.Random(23)
rng.shuffle(filtered_pairs)

# 4. Partición estricta 80% Train / 10% Validation / 10% Test
n = len(filtered_pairs)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_pairs = filtered_pairs[:n_train]
val_pairs   = filtered_pairs[n_train:n_train + n_val]
test_pairs  = filtered_pairs[n_train + n_val:]

# 5. Huella de auditoría (debe reportarse en el notebook y en el PDF)
test_hash = hashlib.sha256("\n".join([p[0] for p in test_pairs]).encode()).hexdigest()[:12]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
print(f"SHA-256 Test Audit ID: {test_hash}")

Verificamos que la implementación modular (`src/data.py`) produce exactamente la misma partición que el bloque del enunciado:

In [ ]:
from src.data import load_and_split
tr2, va2, te2, h2 = load_and_split("spa.txt", verbose=False)
assert (tr2, va2, te2, h2) == (train_pairs, val_pairs, test_pairs, test_hash)
print("OK: src.data.load_and_split reproduce el protocolo. Test Audit ID =", test_hash)

## Estadísticas de longitud y buckets de evaluación (8–18 vs >18 palabras)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from src.data import length_bucket
def stats(pairs, name):
    en = [len(p[0].split()) for p in pairs]; es = [len(p[1].split()) for p in pairs]
    b = pd.Series([length_bucket(p[0]) for p in pairs]).value_counts()
    return {"partición": name, "n": len(pairs), "EN media": round(sum(en)/len(en), 2), "EN máx": max(en),
            "ES media": round(sum(es)/len(es), 2), "ES máx": max(es),
            "medias (8-18)": int(b.get("medium", 0)), "largas (>18)": int(b.get("long", 0))}
df = pd.DataFrame([stats(train_pairs, "train"), stats(val_pairs, "val"), stats(test_pairs, "test")])
display(df)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist([len(p[0].split()) for p in train_pairs], bins=range(8, 60), alpha=.7, label="EN (train)")
ax.hist([len(p[1].split()) for p in train_pairs], bins=range(1, 60), alpha=.5, label="ES (train)")
ax.axvline(18.5, c="k", ls="--", lw=1); ax.set_xlabel("palabras"); ax.set_ylabel("frecuencia"); ax.legend()
ax.set_title("Distribución de longitudes (train)"); fig.tight_layout()
fig.savefig(f"{RESULTS}/longitudes.png", dpi=200)

## Regla 1 · Vocabularios solo con `train_pairs` y mapeo a `<unk>`

In [ ]:
# === 3. Vocabularios SOLO con train_pairs (cero fuga de datos) ================
from src.data import build_vocabs, oov_rate, make_loader
MIN_FREQ = 2          # mismo valor en A, B y C; palabras más raras -> <unk>
eng_word2idx, eng_idx2word, spa_word2idx, spa_idx2word = build_vocabs(train_pairs, min_freq=MIN_FREQ)
print(f"|V_en| = {len(eng_word2idx):,} | |V_es| = {len(spa_word2idx):,}")
print("OOV val :", oov_rate(val_pairs, eng_word2idx, spa_word2idx))
print("OOV test:", oov_rate(test_pairs, eng_word2idx, spa_word2idx))

In [ ]:
from src.data import encode, tokenize
# Ejemplo: una palabra de test que no esté en el vocabulario de train se mapea a <unk> (índice 3)
ex = next((p for p in test_pairs if any(t not in eng_word2idx for t in tokenize(p[0]))), test_pairs[0])
print(ex[0]); print(tokenize(ex[0])); print(encode(ex[0], eng_word2idx))

## Regla 2 · Aislamiento de `test_pairs`
Ningún notebook usa `test_pairs` en el entrenamiento, la selección de época o el early stopping: todo eso
se decide con `val_loss` (A, B, C) o `val_loss`/`val BLEU` (D). `test_pairs` se usa **una sola vez**, al final.

Hay un detalle del corpus que vale la pena reportar. En Tatoeba, una misma oración en inglés puede tener
varias traducciones al español, así que el *shuffle* puede poner la misma fuente en train y en test.
No es fuga de datos según el protocolo (que es fijo para todos), pero sí ayuda a interpretar el BLEU:

In [ ]:
train_src = set(p[0] for p in train_pairs)
dup = sum(p[0] in train_src for p in test_pairs)
print(f"Oraciones de test cuya fuente EN aparece también en train: {dup} ({100*dup/len(test_pairs):.2f}%)")

## Regla 3 · Normalización simétrica para BLEU

In [ ]:
from src.data import normalize
for s in ["¿Por qué no viniste a la reunión de ayer?", "I don't think Tom's ready.", "Él comió el pingüino."]:
    print(f"{s!r:50} -> {normalize(s)!r}")

In [ ]:
from src.utils import save_json
save_json({"train": len(train_pairs), "val": len(val_pairs), "test": len(test_pairs),
           "test_audit_id": test_hash, "test_src_in_train": dup,
           "vocab_en": len(eng_word2idx), "vocab_es": len(spa_word2idx), "min_freq": MIN_FREQ},
          f"{RESULTS}/data_audit.json")